In [ ]:
import os
import json
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import EfficientNetB0, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from PIL import Image

print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs detected: {len(gpus)}')
if gpus:
    for gpu in gpus:
        print(' -', gpu)
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print('⚠️  No GPU found. Go to Settings → Accelerator → GPU T4 x2')

KAGGLE_INPUT   = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')

base = Path('/kaggle/input/datasets/abdallahalidev/plantvillage-dataset')

DATA_DIR = None
for root, dirs, files in os.walk(base):
    root_path = Path(root)
    images = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if images:
        candidate = root_path.parent
        subdirs = [d for d in candidate.iterdir() if d.is_dir()]
        if len(subdirs) >= 5:
            DATA_DIR = candidate
            break

assert DATA_DIR is not None, 'Could not locate class folders inside the dataset'
print(f'DATA_DIR : {DATA_DIR}')
print(f'Classes  : {len(list(DATA_DIR.iterdir()))}')

IMG_SIZE = 160
BATCH_SIZE = 16
HEAD_EPOCHS = 5
FINETUNE_EPOCHS = 10
INITIAL_LR      = 1e-3
FINETUNE_LR     = 1e-5
UNFREEZE_LAYERS = 30
VAL_SPLIT       = 0.15
TEST_SPLIT      = 0.10
RANDOM_SEED     = 42

MODEL_DIR       = KAGGLE_WORKING / 'plant_disease_model'
SPLIT_DIR       = KAGGLE_WORKING / 'dataset_split'
CHECKPOINT_DIR  = MODEL_DIR / 'checkpoints'
LOG_DIR         = MODEL_DIR / 'logs'

for d in [MODEL_DIR, SPLIT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# DATA SPLIT (unchanged)
if (SPLIT_DIR / 'train').exists() and any((SPLIT_DIR / 'train').iterdir()):
    print('Split already exists — skipping copy step')
else:
    for class_dir in sorted(DATA_DIR.iterdir()):
        if not class_dir.is_dir():
            continue

        images = list(class_dir.glob('*.*'))
        if not images:
            continue

        train_imgs, temp = train_test_split(images, test_size=(VAL_SPLIT + TEST_SPLIT), random_state=RANDOM_SEED)
        val_ratio = VAL_SPLIT / (VAL_SPLIT + TEST_SPLIT)
        val_imgs, test_imgs = train_test_split(temp, test_size=(1 - val_ratio), random_state=RANDOM_SEED)

        for split, imgs in [('train', train_imgs), ('val', val_imgs), ('test', test_imgs)]:
            dest = SPLIT_DIR / split / class_dir.name
            dest.mkdir(parents=True, exist_ok=True)
            for img_path in imgs:
                shutil.copy2(img_path, dest / img_path.name)

train_datagen = ImageDataGenerator(rescale=1.0 / 255)
eval_datagen = ImageDataGenerator(rescale=1.0 / 255)

flow_args = dict(target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical')

train_gen = train_datagen.flow_from_directory(str(SPLIT_DIR / 'train'), shuffle=True, **flow_args)
val_gen   = eval_datagen.flow_from_directory(str(SPLIT_DIR / 'val'), shuffle=False, **flow_args)
test_gen  = eval_datagen.flow_from_directory(str(SPLIT_DIR / 'test'), shuffle=False, **flow_args)

NUM_CLASSES  = train_gen.num_classes
CLASS_NAMES  = list(train_gen.class_indices.keys())

# CLASS WEIGHTS
y_train = train_gen.classes
weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
CLASS_WEIGHTS = dict(enumerate(weights))

# 🔥 FIXED MODEL BUILDER
def build_model(num_classes):
    try:
        print("Trying EfficientNetB0 (ImageNet weights)...")
        base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    except:
        print("⚠️ EfficientNet failed. Falling back to MobileNetV2...")
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

    base_model.trainable = False

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=INITIAL_LR),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3)]
    )

    return model

model = build_model(NUM_CLASSES)

# TRAIN
model.fit(train_gen, epochs=HEAD_EPOCHS, validation_data=val_gen, class_weight=CLASS_WEIGHTS)

# FINE-TUNE
base_model = model.layers[1]
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINETUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3)]
)

model.fit(train_gen, epochs=FINETUNE_EPOCHS, validation_data=val_gen, class_weight=CLASS_WEIGHTS)

# EVALUATION
predictions = model.predict(test_gen)
y_pred = np.argmax(predictions, axis=1)
y_true = test_gen.classes

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# ✅ FIXED SAVING
keras_path = str(MODEL_DIR / 'plant_disease_model_final.keras')
saved_model_path = str(MODEL_DIR / 'saved_model')

model.save(keras_path)              # correct
model.export(saved_model_path)      # FIXED (instead of model.save)

print(f'✅ Saved .keras → {keras_path}')
print(f'✅ SavedModel → {saved_model_path}')